In [ ]:
from pathlib import Path
import json                                                                                                                                                  
import re
import numpy as np                                                                                                                                           
                
layer20_base_dir = Path('/mnt/nvme/fjh/glimpse_llava_eval/llava-v1.5-7b')                                                                                    
layer20_run_dirs = {
    'original':       layer20_base_dir / 'original'       / 'token_scores/gqa_rand_seed42_20000',                                                            
    'perturbation_1': layer20_base_dir / 'perturbation_1' / 'token_scores/gqa_rand_seed42_20000',                                                            
    'perturbation_2': layer20_base_dir / 'perturbation_2' / 'token_scores/gqa_rand_seed42_20000',                                                            
    'perturbation_3': layer20_base_dir / 'perturbation_3' / 'token_scores/gqa_rand_seed42_20000',                                                            
    'perturbation_4': layer20_base_dir / 'perturbation_4' / 'token_scores/gqa_rand_seed42_20000',                                                            
}                                                                                                                                                            
layer20_target_layer = 20                                                                                                                                    
layer20_file_glob    = 'gqa_rand_seed42_20000_*.json'                                                                                                        
layer20_output_dir   = layer20_base_dir / 'layer20_token_stats_per_doc'
                                                                                                                                                            
                
def layer20_numeric_suffix(filename: str) -> int:                                                                                                            
    m = re.search(r'(\d+)\.json$', filename)
    return int(m.group(1)) if m else 10**18                                                                                                                  

                                                                                                                                                            
def layer20_get_payload(sample_obj: dict, target_layer: int):
    per_layer_map = sample_obj.get('per_layer_text_to_visual', {}) or {}                                                                                     
    for key, payload in per_layer_map.items():                                                                                                               
        if not isinstance(payload, dict):                                                                                                                    
            continue                                                                                                                                         
        idx = payload.get('layer_index_1based')                                                                                                              
        if idx is None:                                                                                                                                      
            m = re.match(r'layer_(\d+)$', str(key))
            if m:                                                                                                                                            
                idx = int(m.group(1))
        if idx == target_layer:                                                                                                                              
            return payload
    return None

                                                                                                                                                            
def layer20_load_vectors(json_path: Path, target_layer: int):
    with open(json_path, 'r', encoding='utf-8') as f:                                                                                                        
        obj = json.load(f)                                                                                                                                   
    payload = layer20_get_payload(obj, target_layer)
    if payload is None:                                                                                                                                      
        return None
    return np.asarray(payload.get('text_to_visual_similarity', []), dtype=np.float64)
                                                                                                                                                            
                                                                                                                                                            
# ── collect file maps for 5 runs ───────────────────────────────────────────────                                                                            
layer20_run_filemaps = {}                                                                                                                                    
for run_name, run_dir in layer20_run_dirs.items():
    files = sorted(run_dir.glob(layer20_file_glob), key=lambda p: layer20_numeric_suffix(p.name))                                                            
    layer20_run_filemaps[run_name] = {p.name: p for p in files}
    print(f'{run_name:20s}: {len(files)} files')                                                                                                             
                                                                                                                                                            
layer20_common_names = sorted(                                                                                                                               
    set.intersection(*[set(m.keys()) for m in layer20_run_filemaps.values()]),                                                                               
    key=layer20_numeric_suffix,                                                                                                                              
)
print('common files across 5 runs:', len(layer20_common_names))                                                                                              
                                                                                                                                                            
layer20_output_dir.mkdir(parents=True, exist_ok=True)                                                                                                        
layer20_ok      = 0                                                                                                                                          
layer20_skipped = 0                                                                                                                                          
                
for sample_name in layer20_common_names:                                                                                                                     
    simi_runs = []
    for run_name in layer20_run_dirs.keys():                                                                                                                 
        simi_vec = layer20_load_vectors(layer20_run_filemaps[run_name][sample_name], layer20_target_layer)
        if simi_vec is None or simi_vec.size == 0:                                                                                                           
            break                                                                                                                                            
        simi_runs.append(simi_vec)                                                                                                                           
    else:                                                                                                                                                    
        token_len = min(x.size for x in simi_runs)
        simi_stack = np.stack([x[:token_len] for x in simi_runs], axis=0)                                                                                    
        simi_avg   = np.mean(simi_stack, axis=0)                                                                                                             
        simi_var   = np.var(simi_stack,  axis=0)                                                                                                             
                                                                                                                                                            
        m = re.search(r'(\d+)\.json$', sample_name)
        doc_id = int(m.group(1)) if m else None                                                                                                              
                                                                                                                                                            
        token_stats = {
            str(t): {'simi_avg': float(simi_avg[t]), 'simi_var': float(simi_var[t])}                                                                         
            for t in range(token_len)                                                                                                                        
        }
                                                                                                                                                            
        out_obj  = {'doc_id': doc_id, 'token_stats': token_stats}                                                                                            
        out_path = layer20_output_dir / f'{doc_id}.json'
        with open(out_path, 'w', encoding='utf-8') as fout:                                                                                                  
            json.dump(out_obj, fout, ensure_ascii=False)
        layer20_ok += 1                                                                                                                                      
        continue                                                                                                                                             

    layer20_skipped += 1                                                                                                                                     
                
print('output dir :', layer20_output_dir)                                                                                                                    
print('processed  :', layer20_ok)
print('skipped    :', layer20_skipped)    

original      : 20000 files
perturbation_1: 20000 files
perturbation_2: 20000 files
perturbation_3: 20000 files
perturbation_4: 20000 files
common files across 5 runs: 20000
output dir : /mnt/nvme/fjh/glimpse_llava_eval/llava-v1.5-7b/layer14_token_stats_per_doc
processed  : 20000
skipped    : 0
